# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI 協助）
- Date first created: 20261017
- Background: 既有 2026 候選人類型與競選策略提案的資料準備。
- Unit of observation: 名冊及蒐集表每位登記者一列；姓名線索表每筆歷史命中一列。
- Sample period: 2026-09-07 登記名冊；歷史資料 2001–2022。
- Known issues: 尚非最終候選人樣本；姓名命中未核實、早期與其他公職履歷不完整。所有社群、2026 學歷與首次參選研究欄位待蒐集，不填零，不可直接執行完整分群。沒有套用提案尚未確認的排除條件。當前資格審查公告需另逐人核對。
- Definition for each variable: 見下方逐組字典。
- Data path / Source data / Processing code / Role / Verification status / SHA-256 / Downstream analysis input: 見下列逐檔紀錄；來源表與歷史快照見 [raw 文件](../raw/sources_2026_documentation.ipynb)；產生程式為 [prepare_candidates_2026.ipynb](../code/prepare_candidates_2026.ipynb)。role 為 preparation；完成待核對欄位後才能另建立 analysis 版本，不宣稱這批已適合建模。

- `data/processed/candidates_2026_registration.csv`: 81 列，SHA-256 `765797d09de8e70f6cb54486cf660c44f93e7aab1c68c3a6233db60aad7c5832`，重讀/欄位/鍵檢查通過（2026-10-17）。
- `data/processed/candidate_prior_mayor_name_matches.csv`: 25 列，SHA-256 `8a75f370e91438b5eb1462d328baa60473dbae29964f36fbeb48c8aa8e134c4f`，重讀/欄位/鍵檢查通過（2026-10-17）。
- `data/processed/campaign_collection_worklist.csv`: 81 列，SHA-256 `c0986b0282dc2b9911c46f864fff459bef189260db1b0a0fa16c09fe1e5c006d`，重讀/欄位/鍵檢查通過（2026-10-17）。
#### 名冊欄位
- candidate_key：依本版來源順序建立的 2026 ID；非 CEC 號次，更新版須維護對照。
- county_city、candidate_name、office_type、source_note：來源選區、姓名、C1/C2 職務及備註。
- registration_date_roc / registration_date：原民國與 ISO 登記日。
- party_recommendation_original / party_at_registration：原推薦政黨與標準化標籤；不是政黨黨籍。
- source_file、source_page、source_url：官方 PDF 檔、頁次及下載網址。
- metropolitan_2026：六都 1，其餘 0。
- qualification_status、sample_inclusion：待核對審定狀態及待組內決定納入規則。
- sex_2026、birth_date_verified、age_2026_minus_birth_year、education_2026、incumbency_2026：待 2026 來源核對；年齡採 2026 減出生年，與足歲不同。
- first_time_mayor_lifetime：終生首次參選縣市長；first_time_any_elected_office：終生首次參選任何民選公職。1=首次、0=非首次、空白=未知。兩者應分開分析，不同時當成已確認的分群設定。
- history_identity_verified、first_time_review_status：身分及履歷核對狀態。
- prior_mayor_name_match_count_2001_2022 / prior_mayor_name_match_years：姓名命中次數及年份，涵蓋資料包內跨縣市參選；不是已確認個人履歷。

#### 歷史姓名線索欄位
candidate_key 連名冊；candidate_name_2026/county_city_2026 是當次姓名選區；historical_name/county_city/election_date/sex/birth_year/education_cec/education_details/person_key 是歷史來源觀察值，不能當成 2026 學歷或身份確證。source_url/source_file_legacy 是舊包來源；match_status 為僅姓名命中待人工核對。

#### 蒐集工作表欄位
前三欄為候選人鍵、縣市、姓名。official_profile_url / official_facebook_url / official_website_url 為待核對官方頁面；platform_verification_evidence 為官方帳號認證依據；profile_capture_date 為實際擷取日；sex_source_url / birth_source_url / education_source_url / current_office_source_url / prior_mayor_history_source_url / any_office_history_source_url 為各欄證據；bulletin_2026_url 為公報；visual_capture_path 為主視覺留存；follower_count_start/end 為起迄實際觀測追蹤數；reviewer/review_notes 為核對者與註記；collection_status 的 not_collected 表未蒐集，不能解讀為沒有社群。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data
僅讀取以下固定版本；更新資料須核對後修改綁定。

In [2]:
FILES={'candidates_2026_registration': 'data/processed/candidates_2026_registration.csv', 'candidate_prior_mayor_name_matches': 'data/processed/candidate_prior_mayor_name_matches.csv', 'campaign_collection_worklist': 'data/processed/campaign_collection_worklist.csv'}
HASHES={'candidates_2026_registration': '765797d09de8e70f6cb54486cf660c44f93e7aab1c68c3a6233db60aad7c5832', 'candidate_prior_mayor_name_matches': '8a75f370e91438b5eb1462d328baa60473dbae29964f36fbeb48c8aa8e134c4f', 'campaign_collection_worklist': 'c0986b0282dc2b9911c46f864fff459bef189260db1b0a0fa16c09fe1e5c006d'}
SCHEMAS={'candidates_2026_registration': ['candidate_key', 'county_city', 'registration_date_roc', 'candidate_name', 'party_recommendation_original', 'source_note', 'office_type', 'source_file', 'source_page', 'source_url', 'registration_date', 'party_at_registration', 'metropolitan_2026', 'qualification_status', 'sample_inclusion', 'sex_2026', 'birth_date_verified', 'age_2026_minus_birth_year', 'education_2026', 'incumbency_2026', 'first_time_mayor_lifetime', 'first_time_any_elected_office', 'history_identity_verified', 'first_time_review_status', 'prior_mayor_name_match_count_2001_2022', 'prior_mayor_name_match_years'], 'candidate_prior_mayor_name_matches': ['candidate_key', 'candidate_name_2026', 'county_city_2026', 'historical_name', 'historical_county_city', 'historical_election_date', 'historical_sex', 'historical_birth_year', 'historical_education_cec', 'historical_education_details', 'historical_person_key', 'source_url', 'source_file_legacy', 'match_status'], 'campaign_collection_worklist': ['candidate_key', 'county_city', 'candidate_name', 'official_profile_url', 'official_facebook_url', 'official_website_url', 'platform_verification_evidence', 'profile_capture_date', 'sex_source_url', 'birth_source_url', 'education_source_url', 'current_office_source_url', 'prior_mayor_history_source_url', 'any_office_history_source_url', 'bulletin_2026_url', 'visual_capture_path', 'follower_count_start', 'follower_count_end', 'reviewer', 'review_notes', 'collection_status']}
ROWS={'candidates_2026_registration': 81, 'candidate_prior_mayor_name_matches': 25, 'campaign_collection_worklist': 81}

## Summary

In [3]:
frames={}
for key,relative in FILES.items():
    path=project_file(relative,"processed")
    verify_input_sha256(path,HASHES[key])
    frames[key]=pd.read_csv(path,dtype=str,keep_default_na=False)
    print(key,len(frames[key]),"rows")

candidates_2026_registration 81 rows
candidate_prior_mayor_name_matches 25 rows
campaign_collection_worklist 81 rows


### Sample

In [4]:
for key,frame in frames.items():
    print(key); display(frame.head(10))

candidates_2026_registration


,candidate_key,county_city,registration_date_roc,candidate_name,party_recommendation_original,source_note,office_type,source_file,source_page,source_url,...,birth_date_verified,age_2026_minus_birth_year,education_2026,incumbency_2026,first_time_mayor_lifetime,first_time_any_elected_office,history_identity_verified,first_time_review_status,prior_mayor_name_match_count_2001_2022,prior_mayor_name_match_years
0,2026-C1-001,臺北市,115/08/31,郭璽,台灣麻將最大黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
1,2026-C1-002,臺北市,115/09/02,沈伯洋,民主進步黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
2,2026-C1-003,臺北市,115/09/02,蕭文乾,臺灣SoR無法黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
3,2026-C1-004,臺北市,115/09/03,唐新民,三勢團結促進聯盟,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,1,2022
4,2026-C1-005,臺北市,115/09/04,蔣萬安,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,1,2022
5,2026-C1-006,臺北市,115/09/04,林志成,無,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
6,2026-C1-007,新北市,115/09/02,蘇巧慧,民主進步黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
7,2026-C1-008,新北市,115/09/04,李四川,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
8,2026-C1-009,新北市,115/09/04,蘇輝湟,無,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,0,
9,2026-C1-010,桃園市,115/09/01,張善政,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,...,,,,,,,,pending_identity_and_lifetime_history,1,2022


candidate_prior_mayor_name_matches


,candidate_key,candidate_name_2026,county_city_2026,historical_name,historical_county_city,historical_election_date,historical_sex,historical_birth_year,historical_education_cec,historical_education_details,historical_person_key,source_url,source_file_legacy,match_status
0,2026-C1-004,唐新民,臺北市,唐新民,臺北市,2022-11-26,男,1950,博士,• 大同初中\n\n• 建國高中\n\n• 台灣大學土木系學士\n\n• 柏克萊加大電機計算...,唐新民|男|1950,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review
1,2026-C1-005,蔣萬安,臺北市,蔣萬安,臺北市,2022-11-26,男,1978,博士,• 師大附中國中部\n\n• 台北市立建國高級中學\n\n• 政治大學法律系及外交系雙\n\...,蔣萬安|男|1978,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review
2,2026-C1-010,張善政,桃園市,張善政,桃園市,2022-11-26,男,1954,博士,美國康乃爾大學土木與環境工程學博士\n美國史丹福大學土木工程學碩士\n國立臺灣大學土木工程學...,張善政|男|1954,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review
3,2026-C1-015,謝龍介,臺南市,謝龍介,臺南市,2022-11-26,男,1961,碩士,國立成功大學管理學院高階\n管理碩士、遠東科技大學、\n臺南二中、長榮中學、民德\n國中、立人國小,謝龍介|男|1961,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review
4,2026-C1-022,柯志恩,高雄市,柯志恩,高雄市,2022-11-26,女,1962,博士,美國南加大教育心理學博士,柯志恩|女|1962,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review
5,2026-C2-024,鄭朝方,新竹縣,鄭朝方,新竹縣,2018-11-24,男,1980,大學,一、臺北大學財經法律系\n二、新竹中學\n三、竹東國中\n四、大同國小、竹東國小,鄭朝方|男|1980,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2018_C2_f75e94cad2958f8764f9d09...,name_only_pending_identity_review
6,2026-C2-026,徐欣瑩,新竹縣,徐欣瑩,新竹縣,2018-11-24,女,1972,博士,國立交通大學土木系博士、碩士\n國立成功大學測量系學士\n中山女中\n新豐國中\n山崎國小,徐欣瑩|女|1972,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2018_C2_f75e94cad2958f8764f9d09...,name_only_pending_identity_review
7,2026-C2-028,鍾東錦,苗栗縣,鍾東錦,苗栗縣,2022-11-26,男,1963,大學,南埔國民小學\n興華國民中學\n大成高級中學\n中華大學企業管理系,鍾東錦|男|1963,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C2_63615098f5afa8ec53159c4...,name_only_pending_identity_review
8,2026-C2-034,許淑華,南投縣,許淑華,南投縣,2022-11-26,女,1975,碩士,逢甲大學經營管理碩士畢\n南開技術學院附設專科進\n修學校畢,許淑華|女|1975,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C2_63615098f5afa8ec53159c4...,name_only_pending_identity_review
9,2026-C2-035,吳炳輝,雲林縣,吳炳輝,臺南市,2022-11-26,男,1967,大學,1.天主教梵帝崗幼兒園2.土\n庫國民小學3.天主教永年初\n級中學4.板橋高中5.私立東\...,吳炳輝|男|1967,https://db.cec.gov.tw/static/elections/data/ti...,raw/cec/events/2022_C1_05cc7b904c7a30cc7c88d5b...,name_only_pending_identity_review


campaign_collection_worklist


,candidate_key,county_city,candidate_name,official_profile_url,official_facebook_url,official_website_url,platform_verification_evidence,profile_capture_date,sex_source_url,birth_source_url,...,current_office_source_url,prior_mayor_history_source_url,any_office_history_source_url,bulletin_2026_url,visual_capture_path,follower_count_start,follower_count_end,reviewer,review_notes,collection_status
0,2026-C1-001,臺北市,郭璽,,,,,,,,...,,,,,,,,,,not_collected
1,2026-C1-002,臺北市,沈伯洋,,,,,,,,...,,,,,,,,,,not_collected
2,2026-C1-003,臺北市,蕭文乾,,,,,,,,...,,,,,,,,,,not_collected
3,2026-C1-004,臺北市,唐新民,,,,,,,,...,,,,,,,,,,not_collected
4,2026-C1-005,臺北市,蔣萬安,,,,,,,,...,,,,,,,,,,not_collected
5,2026-C1-006,臺北市,林志成,,,,,,,,...,,,,,,,,,,not_collected
6,2026-C1-007,新北市,蘇巧慧,,,,,,,,...,,,,,,,,,,not_collected
7,2026-C1-008,新北市,李四川,,,,,,,,...,,,,,,,,,,not_collected
8,2026-C1-009,新北市,蘇輝湟,,,,,,,,...,,,,,,,,,,not_collected
9,2026-C1-010,桃園市,張善政,,,,,,,,...,,,,,,,,,,not_collected


### Summary Stats

In [5]:
for key,frame in frames.items():
    print(key); display((frame=="").sum().rename("blank_cells").to_frame())
display(frames["candidates_2026_registration"].groupby("county_city").size().rename("registrants").to_frame())

candidates_2026_registration


,blank_cells
candidate_key,0
county_city,0
registration_date_roc,0
candidate_name,0
party_recommendation_original,0
source_note,81
office_type,0
source_file,0
source_page,0
source_url,0


candidate_prior_mayor_name_matches


,blank_cells
candidate_key,0
candidate_name_2026,0
county_city_2026,0
historical_name,0
historical_county_city,0
historical_election_date,0
historical_sex,0
historical_birth_year,0
historical_education_cec,1
historical_education_details,8


campaign_collection_worklist


,blank_cells
candidate_key,0
county_city,0
candidate_name,0
official_profile_url,81
official_facebook_url,81
official_website_url,81
platform_verification_evidence,81
profile_capture_date,81
sex_source_url,81
birth_source_url,81


,registrants
county_city,
南投縣,2
嘉義市,5
嘉義縣,2
基隆市,3
宜蘭縣,5
屏東縣,2
彰化縣,4
新北市,3
新竹市,3


### Assertions

In [6]:
for key,frame in frames.items():
    assert list(frame.columns)==SCHEMAS[key] and len(frame)==ROWS[key]
    assert not frame.duplicated().any()
    if key!="candidate_prior_mayor_name_matches": assert frame.candidate_key.is_unique
keys=set(frames["candidates_2026_registration"].candidate_key)
assert frames["candidates_2026_registration"].county_city.nunique()==22
for frame in frames.values(): assert set(frame.candidate_key)<=keys
assert (frames["candidates_2026_registration"][["first_time_mayor_lifetime","first_time_any_elected_office"]]=="").all().all()
print("All configured checks passed. Research fields remain pending, not zero.")


All configured checks passed. Research fields remain pending, not zero.
